In [17]:
import os
import duckdb
import polars as pl

DATA_PATH = "../data/raw/Border_Crossing_Entry_Data.csv"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Path dataset: {DATA_PATH}")
print(f"Status file: {'Ditemukan' if os.path.exists(DATA_PATH) else 'TIDAK DITEMUKAN - Cek path!'}")

Path dataset: ../data/raw/Border_Crossing_Entry_Data.csv
Status file: Ditemukan


In [16]:
# Membaca data secara lazy (LazyFrame)
lazy_df = pl.scan_csv(DATA_PATH)

# Menggunakan collect_schema() 
schema = lazy_df.collect_schema()
print("=== SKEMA DATASET ===")
for col_name, dtype in schema.items():
    print(f"- {col_name}: {dtype}")

# Menghitung baris dan kolom 
total_rows = lazy_df.select(pl.len()).collect().item()
total_cols = len(schema)

print(f"\nTotal Baris : {total_rows:,}")
print(f"Total Kolom : {total_cols}")

=== SKEMA DATASET ===
- Port Name: String
- State: String
- Port Code: Int64
- Border: String
- Date: String
- Measure: String
- Value: Int64
- Location: String

Total Baris : 346,733
Total Kolom : 8


In [9]:
print("5 Baris Pertama:")
display(lazy_df.head(5).collect())

print("5 Baris Terakhir:")
display(lazy_df.tail(5).collect())

5 Baris Pertama:


Port Name,State,Port Code,Border,Date,Measure,Value,Location
str,str,i64,str,str,str,i64,str
"""Calexico East""","""California""",2507,"""US-Mexico Border""","""03/01/2019 12:00:00 AM""","""Trucks""",34447,"""POINT (-115.48433000000001 32.…"
"""Van Buren""","""Maine""",108,"""US-Canada Border""","""03/01/2019 12:00:00 AM""","""Rail Containers Full""",428,"""POINT (-67.94271 47.16207)"""
"""Otay Mesa""","""California""",2506,"""US-Mexico Border""","""03/01/2019 12:00:00 AM""","""Trucks""",81217,"""POINT (-117.05333 32.57333)"""
"""Nogales""","""Arizona""",2604,"""US-Mexico Border""","""03/01/2019 12:00:00 AM""","""Trains""",62,"""POINT (-110.93361 31.340279999…"
"""Trout River""","""New York""",715,"""US-Canada Border""","""03/01/2019 12:00:00 AM""","""Personal Vehicle Passengers""",16377,"""POINT (-73.44253 44.9900100000…"


5 Baris Terakhir:


Port Name,State,Port Code,Border,Date,Measure,Value,Location
str,str,i64,str,str,str,i64,str
"""Norton""","""Vermont""",211,"""US-Canada Border""","""01/01/1996 12:00:00 AM""","""Personal Vehicle Passengers""",9119,"""POINT (-71.79 45.01)"""
"""Columbus""","""New Mexico""",2406,"""US-Mexico Border""","""01/01/1996 12:00:00 AM""","""Personal Vehicles""",29486,"""POINT (-107.63 31.78)"""
"""Hansboro""","""North Dakota""",3415,"""US-Canada Border""","""01/01/1996 12:00:00 AM""","""Rail Containers Full""",0,"""POINT (-99.35 49)"""
"""Calexico East""","""California""",2507,"""US-Mexico Border""","""01/01/1996 12:00:00 AM""","""Buses""",0,"""POINT (-115.39 32.67)"""
"""Presidio""","""Texas""",2403,"""US-Mexico Border""","""01/01/1996 12:00:00 AM""","""Trucks""",347,"""POINT (-104.39000000000001 29.…"


In [15]:
con = duckdb.connect()

# Profiling statistik dengan DuckDB  membaca file CSV
summary_df = con.sql(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')").pl()
display(summary_df)

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""Port Name""","""VARCHAR""","""Alcan""","""Willow Creek""",126,null,null,null,null,null,346733,0.00
"""State""","""VARCHAR""","""Alaska""","""Washington""",16,null,null,null,null,null,346733,0.00
"""Port Code""","""BIGINT""","""101""","""4105""",115,"""2455.5049101181603""","""1201.2084587195827""","""2304""","""3013""","""3396""",346733,0.00
"""Border""","""VARCHAR""","""US-Canada Border""","""US-Mexico Border""",2,null,null,null,null,null,346733,0.00
"""Date""","""TIMESTAMP""","""1996-01-01 00:00:00""","""2019-03-01 00:00:00""",291,"""2007-04-10 23:19:31.461615""",null,"""2001-10-17 04:04:24.691926""","""2007-03-25 08:26:53.027294""","""2012-09-06 03:33:56.283673""",346733,0.00
"""Measure""","""VARCHAR""","""Bus Passengers""","""Trucks""",12,null,null,null,null,null,346733,0.00
"""Value""","""BIGINT""","""0""","""4447374""",46280,"""28187.668442288446""","""151858.8031861271""","""0""","""91""","""2489""",346733,0.00
"""Location""","""VARCHAR""","""POINT (-100.05 49)""","""POINT (-99.71915 48.94208)""",230,null,null,null,null,null,346733,0.00


In [13]:
# 1. Cek persentase null tiap kolom menggunakan Polars expressions
null_audit = lazy_df.select([
    (pl.col(col).null_count() / pl.len() * 100).round(2).alias(f"{col}_null_pct")
    for col in schema.keys()
]).collect()

print("=== PERSENTASE NULL PER KOLOM (%) ===")
display(null_audit)

# 2. Cek baris duplikat penuh
total_unique = lazy_df.unique().select(pl.len()).collect().item()
duplicate_count = total_rows - total_unique

print(f"\nTotal Baris Unik     : {total_unique:,}")
print(f"Jumlah Baris Duplikat: {duplicate_count:,} ({(duplicate_count / total_rows) * 100:.2f}%)")

=== PERSENTASE NULL PER KOLOM (%) ===


Port Name_null_pct,State_null_pct,Port Code_null_pct,Border_null_pct,Date_null_pct,Measure_null_pct,Value_null_pct,Location_null_pct
f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Total Baris Unik     : 346,733
Jumlah Baris Duplikat: 0 (0.00%)


In [14]:
PARQUET_PATH = "../data/processed/raw_data.parquet"

# Menyimpan langsung dari lazy plan tanpa memakan RAM besar
lazy_df.sink_parquet(PARQUET_PATH)

file_size_csv = os.path.getsize(DATA_PATH) / (1024 * 1024)
file_size_parquet = os.path.getsize(PARQUET_PATH) / (1024 * 1024)

print(f"File Parquet berhasil dibuat di: {PARQUET_PATH}")
print(f"Ukuran CSV Asli    : {file_size_csv:.2f} MB")
print(f"Ukuran Parquet Baru: {file_size_parquet:.2f} MB")
print(f"Efisiensi Kompresi : {(1 - file_size_parquet / file_size_csv) * 100:.2f}% lebih hemat")

File Parquet berhasil dibuat di: ../data/processed/raw_data.parquet
Ukuran CSV Asli    : 35.34 MB
Ukuran Parquet Baru: 1.97 MB
Efisiensi Kompresi : 94.43% lebih hemat
